In [1]:
import json
import os
from langchain.llms import Ollama
from langchain.text_splitter import CharacterTextSplitter
import openai
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
import re
import unicodedata
from html import unescape

In [2]:

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 


llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)


In [3]:
# Initialize OpenAI client (new method)
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def ask_openai(question, model="gpt-4o"):
    """Sends a question to OpenAI's API and returns the response."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# Example Usage
question = "What is differential privacy?"
answer = ask_openai(question)
print("OpenAI Response:", answer)


OpenAI Response: Differential privacy is a mathematical framework designed to enable the analysis of data while preserving the privacy of individuals in the dataset. It provides a formal definition of privacy that allows data analysts to extract useful information from large datasets without revealing sensitive information about any individual in the dataset.

The key idea behind differential privacy is to add a controlled amount of random noise to the results of queries made on the dataset. This noise ensures that the presence or absence of any single individual's data has a minimal impact on the output of any analysis, thereby protecting individual privacy. The amount of noise added is calibrated according to a privacy parameter, often denoted as \( \varepsilon \) (epsilon), which quantifies the privacy guarantee: smaller values of epsilon provide stronger privacy guarantees but may result in less accurate query results.

Differential privacy is widely used because it provides strong

In [4]:
ollama = Ollama(base_url='http://localhost:11434', model="llama3.1:70b")

/tmp/ipykernel_3426690/2524497889.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434', model="llama3.1:70b")


In [5]:
# system_prompt = """
# # Hybrid Knowledge Graph and Keyword Extraction Agent

# ## Role
# You are an advanced information extraction agent specialized in building comprehensive knowledge graphs and keyword indexes from unstructured text. Your output is used for semantic search, indexing, reasoning, and data validation.

# ## Objective
# From the provided text, extract:
# 1. **Entities as nodes** with relevant attributes.
# 2. **Relationships between entities** with relevant attributes.
# 3. **High-quality keywords as nodes**, directly linked to the document for fast indexing.
# 4. **The document node must store the content of the document itself.**

# All outputs must be included in a **single, valid JSON object** following the defined structure.

# ## Allowed Labels
# Each keyword node must use one label from the following list:

# { "person", "organization", "location", "event", "date", "work", "law", "product", "language", "scientific_term", "other" }

# If a term doesn't fit any category clearly, use `"other"`.

# ---

# ## Output Format (JSON)
# ```json
# {
#   "nodes": [
#     {
#       "id": "unique_node_id",
#       "label": "nodetype",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "id": "unique_keyword_id",
#       "label": "keyword_label"
#     },
#     {
#       "id": "docX",
#       "label": "document",
#       "attributes": {
#         "content": "Original document content here."
#       }
#     }
#   ],
#   "relationships": [
#     {
#       "source": "source_node_id",
#       "target": "target_node_id",
#       "type": "RELATIONSHIP_TYPE",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "source": "unique_keyword_id",
#       "target": "docX",
#       "type": "MENTIONED_IN"
#     }
#   ]
# }

# """


In [6]:
system_prompt = """
# Hybrid Knowledge Graph and Keyword Extraction Agent

## Role
You are an advanced information extraction agent specialized in building comprehensive knowledge graphs and keyword indexes from unstructured text. Your output is used for semantic search, indexing, reasoning, and data validation.

---

## Objective
From the provided text, extract:
1. **Entities as nodes** with relevant attributes.
2. **Relationships between entities** with relevant attributes.

Return results as a **single, valid JSON object** following the defined structure.

---

## Allowed Labels
Each non-document node must use **one** label from the following list:

{ "person", "entertainment", "organization", "location", "event", "work", "law", "product", "language", "scientific_term", "other" }

> Note:
> - If a term doesn't clearly fit any category, use `"other"`.
> - Do **not** create separate date nodes; store dates as attributes (see “Numbers and Dates”).

---

## Output Format (JSON)

{
  "nodes": [
    {
      "id": "unique_node_id",
      "label": "allowed_label",
      "doc_id": "DOC_ID",
      "attributes": {
        "key1": "value1",
        "key2": "value2"
      }
    },
    {
      "id": "unique_keyword_id",
      "label": "allowed_label_or_other",
      "doc_id": "DOC_ID"
    },
    
  ],
  "relationships": [
    {
      "source": "source_node_id",
      "target": "target_node_id",
      "type": "RELATIONSHIP_TYPE",
      "attributes": {
        "key1": "value1"
      }
    }
  ]
}

---

## Extraction Guidelines

### 1) Entity Nodes
Extract persons, organizations, locations, events, products, laws, scientific terms, works (creative outputs), and other concepts.
- **Attributes:** Use camelCase keys (e.g., `name`, `role`, `episodeCount`, `releaseDate`, `headquartersCity`).
- **IDs:** Lowercase, singular, underscore-separated (e.g., `srushti_bhavsar`, `chicago_fire_season_4`).
- **Labels:** Choose exactly one from the Allowed Labels list.

### 2) Label Assignment Examples (Corrected)
- `Srushti Bhavsar` → `person`
- `Chicago Fire Season 4` → `entertainment`
- `European Commission` → `organization`
- `Berlin` → `location`
- `GDPR` → `law`
- `World Cup 2014` → `event`
- `CRISPR-Cas9` → `scientific_term`
- `Spanish` → `language`
- `iPhone 15 Pro` → `product`
- Unclear category → `other`


> Do **not** use `"docX"`; that was only an example placeholder.

### 3) Relationships
- Use **ALL CAPS** for relationship `type` values (e.g., `PRODUCED`, `LOCATED_IN`, `BASED_ON`, `PART_OF`, `AFFILIATED_WITH`).
- Include relationship attributes when useful (camelCase keys).
- **MANDATORY:** Every non-document node (entities and keywords) must have exactly one `MENTIONED_IN` relationship **to** the document node (`doc_DOC_ID`). Do **not** include attributes on `MENTIONED_IN`.

### 4) Numbers and Dates
- **Dates:** Store as attributes in **YYYY-MM-DD** format where possible (e.g., `releaseDate`, `startDate`, `endDate`). Do **not** create date nodes.
- **Numbers:** Store as attributes with descriptive keys (e.g., `episodeCount`: 23). For large/approximate values, use strings (e.g., `"ownershipPercentage": "less than 50"`).


### 5) Strict Compliance
- Output **valid JSON only**. No markdown, comments, or trailing text.
- If **no** data can be extracted, return exactly:
  { "nodes": [], "relationships": [] }
"""


In [7]:
# Function to read text from a file
def read_text_from_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


In [8]:
def sanitize_paragraph(text: str, max_chars: int | None = None) -> str:
    if not text:
        return ""
    # 1) Unicode normalize + unescape HTML entities
    t = unicodedata.normalize("NFKC", text)
    t = unescape(t)

    # 2) Fix common mojibake: "60Â°" -> "60°"
    t = t.replace("Â°", "°")

    # 3) Collapse whitespace early
    t = re.sub(r"\s+", " ", t).strip()

    # 4) Remove stray digits placed before a LaTeX fraction (e.g., "3 2 \tfrac{...}{...}")
    t = re.sub(r"\b\d+\s+\d+\s+(?=\\tfrac\b)", "", t)

    # 5) Remove LaTeX display command
    t = re.sub(r"\\displaystyle\b", "", t)

    # 6) Convert \sqrt{...} -> sqrt(...)
    def _sqrt(m): return f"sqrt({m.group(1)})"
    t = re.sub(r"\\sqrt\s*\{([^}]*)\}", _sqrt, t)

    # 7) Convert \tfrac{num}{den} -> num/den  (works with nested sqrt already converted)
    def _tfrac(m):
        num = m.group(1).strip()
        den = m.group(2).strip()
        return f"{num}/{den}"
    t = re.sub(r"\\tfrac\s*\{([^}]*)\}\s*\{([^}]*)\}", _tfrac, t)

    # 8) Remove remaining LaTeX braces/backslashes
    t = t.replace("{", "").replace("}", "").replace("\\", "")

    # 9) Fix spaces before punctuation
    t = re.sub(r"\s+([.,;:!?])", r"\1", t)

    # 10) Lowercase and remove apostrophes
    t = t.lower().replace("'", "")
    
    t = t.replace('"', '')


    # 11) Optional length cap
    if max_chars and len(t) > max_chars:
        t = t[:max_chars].rstrip()

    return t


In [9]:
def extract_knowledge_graph(input_text_chunk, doc_id, title,labels_list):
    prompt = f"{system_prompt}\n\nDocument ID:\n{doc_id}\n\nTitle:\n{title}\n\nInput Text:\n{input_text_chunk}"
    response = ask_openai(prompt)
    # Clean up markdown code block markers if present
    if response.strip().startswith("```"):
        response = response.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    
    return response



In [10]:
def process_single_document(doc, combined_graph, labels_list, failed_chunks_file, max_retries=2):
    doc_id = doc["_id"]
    text = doc["text"]
    retries = 0

    while retries < max_retries:
        try:
            response = extract_knowledge_graph(text, doc_id, list(labels_list))
            extracted_graph = json.loads(response)

            for node in extracted_graph.get("nodes", []):
                if "label" in node:
                    labels_list.add(node["label"])
                if node not in combined_graph["nodes"]:
                    combined_graph["nodes"].append(node)

            for rel in extracted_graph.get("relationships", []):
                if rel not in combined_graph["relationships"]:
                    combined_graph["relationships"].append(rel)


            break  # success
        except json.JSONDecodeError as e:
            retries += 1
            print(f"[ERROR] JSONDecodeError on doc {doc_id}, retry {retries}/{max_retries}: {e}")
            if retries >= max_retries:
                with open(failed_chunks_file, "r+", encoding="utf-8") as f:
                    failed_responses = json.load(f)
                    failed_responses.append({
                        "doc_id": doc_id,
                        "text": text,
                        "response": response
                    })
                    f.seek(0)
                    json.dump(failed_responses, f, indent=4)


In [11]:
def process_jsonl_file(input_path, output_path, failed_chunks_file):
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            combined_graph = json.load(f)
    else:
        combined_graph = {"nodes": [], "relationships": []}

    if not os.path.exists(failed_chunks_file):
        with open(failed_chunks_file, "w", encoding="utf-8") as f:
            json.dump([], f, indent=4)

    labels_list = {node["label"] for node in combined_graph["nodes"] if "label" in node}

    with open(input_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                doc = json.loads(line)
                print(f"\nProcessing document {line_num}: {doc.get('_id')}")
                process_single_document(doc, combined_graph, labels_list, failed_chunks_file)
                with open(output_path, "w", encoding="utf-8") as out_f:
                    json.dump(combined_graph, out_f, indent=4)
            except Exception as e:
                print(f"[ERROR] Failed to process document {line_num}: {e}")

    print(f"\nCompleted. Graph saved at {output_path}.")

In [12]:
# File paths
input_file_path = "/home/sbhavsar/PoisonedRAG/datasets/nq/hybrid.jsonl"
output_file_path = "/home/sbhavsar/PoisonedRAG/build_again/one_by_one/15283_17k.json"
failed_chunks_file_path = "/home/sbhavsar/PoisonedRAG/build_again/one_by_one/failed_15283_17k.json"

# Run the processor
process_jsonl_file(input_file_path, output_file_path, failed_chunks_file_path)

FileNotFoundError: [Errno 2] No such file or directory: '/home/sbhavsar/PoisonedRAG/build_again/one_by_one/failed_15283_17k.json'